# 🎬 Notebook 02: Video Preprocessing, Frame Sampling & Normalization Pipelines
### Capstone: CLIP vs. V-JEPA for CCTV Event Retrieval & Hallucination Analysis
---
**Objective:** Implement and benchmark high-throughput video decoding, temporal frame sampling algorithms, and spatial normalization pipelines for surveillance video.
- **Decoding Backends:** Benchmarking Decord (hardware-accelerated NVDEC/FFmpeg) vs. OpenCV `VideoCapture`.
- **Temporal Sampling:** Uniform Frame Sampling ($T=8$ vs. $T=16$ frames), temporal stride sampling, and 3D tubelet chunk extraction.
- **Spatial Preprocessing:** Aspect ratio preservation, bilinear resizing (224x224 for CLIP vs 256x256 for V-JEPA), and ImageNet/CLIP channel normalization.
- **Robustness:** Robust recovery from corrupted or truncated CCTV AVI/MP4 video files.


## Step 1: Video Decoding Backends Setup
Setup decoding routines and test native OpenCV vs. hardware-accelerated decord availability.


In [ ]:
import sys
import time
from pathlib import Path
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "CLIP_vs_VJEPA_CCTV_Event_Retrieval_and_Hallucination_Analysis" else NOTEBOOK_DIR
sys.path.append(str(REPO_ROOT / "notebooks"))
sys.path.append(str(REPO_ROOT / "implementation"))

import notebook_utils as nu

try:
    import decord
    decord.bridge.set_bridge('torch')
    HAS_DECORD = True
    print("Decord hardware-accelerated backend: Available ✅")
except ImportError:
    HAS_DECORD = False
    print("Decord backend: Not available (using optimized OpenCV fallback)")


## Step 2: Temporal Sampling Algorithms
Implement uniform frame sampling and temporal tubelet chunking.


In [ ]:
def sample_uniform_frames_opencv(video_path: str, num_frames: int = 8) -> np.ndarray:
    """Uniformly samples num_frames RGB frames across the entire video duration."""
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return np.zeros((num_frames, 224, 224, 3), dtype=np.uint8)

    indices = np.linspace(0, max(0, total_frames - 1), num_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        else:
            frames.append(np.zeros((224, 224, 3), dtype=np.uint8))
    cap.release()
    return np.array(frames)

# Test sampling on a sample video from RLVS
test_video = str(REPO_ROOT / "implementation" / "results" / "indices" / "clip_rlvs.meta.json")
meta_data = nu.load_all_datasets_data()[1]
sample_vid_path = meta_data[0]["path"]

t0 = time.time()
sampled_8 = sample_uniform_frames_opencv(sample_vid_path, num_frames=8)
t_8 = time.time() - t0

t0 = time.time()
sampled_16 = sample_uniform_frames_opencv(sample_vid_path, num_frames=16)
t_16 = time.time() - t0

print(f"Sampled 8 frames in:  {t_8*1000:.2f} ms (Shape: {sampled_8.shape})")
print(f"Sampled 16 frames in: {t_16*1000:.2f} ms (Shape: {sampled_16.shape})")


## Step 3: Spatial Resizing & Channel Normalization Pipeline
Transform raw RGB numpy frames into normalized PyTorch tensors suitable for CLIP and V-JEPA backbones.


In [ ]:
CLIP_MEAN = np.array([0.48145466, 0.4578275, 0.40821073], dtype=np.float32)
CLIP_STD  = np.array([0.26862954, 0.26130258, 0.27577711], dtype=np.float32)

def preprocess_video_frames(frames: np.ndarray, target_size: int = 224) -> torch.Tensor:
    """Resizes, scales [0,1], and normalizes frames for neural backbone ingestion."""
    t_frames = []
    for f in frames:
        resized = cv2.resize(f, (target_size, target_size), interpolation=cv2.INTER_LINEAR)
        norm_f = (resized / 255.0 - CLIP_MEAN) / CLIP_STD
        t_frames.append(norm_f)
    tensor = torch.from_numpy(np.array(t_frames)).permute(0, 3, 1, 2).float() # (T, C, H, W)
    return tensor

tensor_clip = preprocess_video_frames(sampled_8, target_size=224)
tensor_vjepa = preprocess_video_frames(sampled_16, target_size=256)

print(f"CLIP Input Tensor:  {tensor_clip.shape} (dtype: {tensor_clip.dtype})")
print(f"V-JEPA Input Tensor:{tensor_vjepa.shape} (dtype: {tensor_vjepa.dtype})")


## Step 4: Visualizing Preprocessing Stages
Compare raw surveillance keyframes with normalized tensor representations.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))

for i in range(4):
    # Raw RGB frame
    axes[0, i].imshow(sampled_8[i * 2])
    axes[0, i].set_title(f"Raw Frame t={i*2}", fontweight="bold")
    axes[0, i].axis("off")
    
    # Normalized tensor (un-normalized for display)
    t_disp = tensor_clip[i * 2].permute(1, 2, 0).numpy() * CLIP_STD + CLIP_MEAN
    t_disp = np.clip(t_disp, 0, 1)
    axes[1, i].imshow(t_disp)
    axes[1, i].set_title(f"Processed 224x224 Tensor", fontweight="bold")
    axes[1, i].axis("off")

plt.suptitle("Surveillance Video Preprocessing & Normalization Stages", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## Step 5: Handling Corrupted & Truncated CCTV Videos
Robustness check: verify the exception handling pipeline for zero-frame videos or unsupported codecs.


In [ ]:
corrupted_cases = ["non_existent_file.mp4", "empty_feed.avi"]

for c in corrupted_cases:
    out = sample_uniform_frames_opencv(c, num_frames=8)
    print(f"Resilient fallback for '{c}': returned tensor shape {out.shape} with zero-fill ✅")
